# Step 1 — Success Pattern Discovery
**Talent Match Intelligence | Case Study DA 2025**

Goal: Identify what differentiates Rating-5 employees across competencies, psychometrics, behavioral data, and contextual factors. Synthesize findings into a weighted Success Formula.

**Actual ERD schema (profiles_psych):** `iq`, `gtq` (int), `tiki` (int), `pauli`, `faxtor`, `disc`, `disc_word`, `mbti`

In [1]:
import os, sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from supabase import create_client
from dotenv import load_dotenv

load_dotenv('../.env')

url = os.environ['SUPABASE_URL']
key = os.environ['SUPABASE_KEY']
sb  = create_client(url, key)

def fetch(table, cols='*', limit=50000):
    r = sb.table(table).select(cols).limit(limit).execute()
    return pd.DataFrame(r.data)

print('Connected to Supabase ✓')

Connected to Supabase ✓


## 0. Data Inventory — Row Counts & Columns

In [2]:
tables = [
    'employees', 'profiles_psych', 'papi_scores', 'strengths',
    'performance_yearly', 'competencies_yearly',
    'dim_companies', 'dim_grades', 'dim_positions', 'dim_directorates',
    'dim_departments', 'dim_divisions', 'dim_education', 'dim_majors',
    'dim_competency_pillars'
]

frames = {}
for t in tables:
    try:
        df = fetch(t)
        frames[t] = df
        print(f'{t:35s} | {len(df):>6} rows | cols: {list(df.columns)}')
    except Exception as e:
        print(f'{t:35s} | ERROR: {e}')

employees                           |   1000 rows | cols: ['employee_id', 'fullname', 'nip', 'company_id', 'area_id', 'position_id', 'department_id', 'division_id', 'directorate_id', 'grade_id', 'education_id', 'major_id', 'years_of_service_months']
profiles_psych                      |   1000 rows | cols: ['employee_id', 'pauli', 'faxtor', 'disc', 'disc_word', 'mbti', 'iq', 'gtq', 'tiki']
papi_scores                         |   1000 rows | cols: ['employee_id', 'scale_code', 'score']
strengths                           |   1000 rows | cols: ['employee_id', 'rank', 'theme']
performance_yearly                  |   1000 rows | cols: ['employee_id', 'year', 'rating']
competencies_yearly                 |   1000 rows | cols: ['employee_id', 'pillar_code', 'year', 'score']
dim_companies                       |      4 rows | cols: ['company_id', 'name']
dim_grades                          |      3 rows | cols: ['grade_id', 'name']
dim_positions                       |      6 rows | cols: ['p

## 1. Load Core Tables

In [3]:
employees     = frames['employees']
perf          = frames['performance_yearly']
psych         = frames['profiles_psych']
papi          = frames['papi_scores']
strengths_df  = frames['strengths']
comp_yearly   = frames['competencies_yearly']
dim_pillars   = frames['dim_competency_pillars']
dim_grades    = frames['dim_grades']
dim_positions = frames['dim_positions']
dim_dirs      = frames['dim_directorates']
dim_edu       = frames['dim_education']

# Inspect actual psych columns
print('profiles_psych columns:', psych.columns.tolist())
print(psych.head(3))

profiles_psych columns: ['employee_id', 'pauli', 'faxtor', 'disc', 'disc_word', 'mbti', 'iq', 'gtq', 'tiki']
  employee_id  pauli  faxtor disc               disc_word  mbti     iq   gtq  \
0   EMP100000     86      75   SI   Steadiness-Influencer  None   94.0  33.0   
1   EMP100001     48      52   DS     Dominant-Steadiness  INTP   94.0  17.0   
2   EMP100002     66      38   DC  Dominant-Conscientious  None  109.0  20.0   

   tiki  
0     2  
1     3  
2     3  


## 2. Null Analysis

In [4]:
print('--- employees ---')
print(employees.isnull().sum())
print('\n--- profiles_psych ---')
print(psych.isnull().sum())
print('\n--- papi_scores ---')
print(papi['scale_code'].value_counts())

--- employees ---
employee_id                0
fullname                   0
nip                        0
company_id                 0
area_id                    0
position_id                0
department_id              0
division_id                0
directorate_id             0
grade_id                   0
education_id               0
major_id                   0
years_of_service_months    0
dtype: int64

--- profiles_psych ---
employee_id      0
pauli            0
faxtor           0
disc            82
disc_word        0
mbti            66
iq             220
gtq            158
tiki             0
dtype: int64

--- papi_scores ---
scale_code
Papi_N    1000
Name: count, dtype: int64


## 3. Define High Performers (Rating = 5)

In [5]:
# Use most recent year per employee
latest_perf = (
    perf.sort_values('year', ascending=False)
        .drop_duplicates('employee_id')
        .rename(columns={'rating': 'latest_rating'})
)

print('Rating distribution:')
print(latest_perf['latest_rating'].value_counts().sort_index())

high_performers = latest_perf[latest_perf['latest_rating'] == 5]['employee_id'].tolist()
print(f'\nHigh performers (rating=5): {len(high_performers)}')
print(f'Others                    : {len(latest_perf) - len(high_performers)}')

Rating distribution:
latest_rating
0.0       3
1.0      53
2.0     150
3.0     296
4.0     207
5.0      56
6.0       1
99.0      1
Name: count, dtype: int64

High performers (rating=5): 56
Others                    : 944


## 4. Rating Distribution

In [6]:
fig = px.histogram(
    latest_perf, x='latest_rating',
    title='Performance Rating Distribution (Latest Year)',
    color_discrete_sequence=['#1d6fa8'],
    labels={'latest_rating': 'Rating'}
)
fig.show()
fig.write_image('../assets/charts/01_rating_distribution.png')

## 5. Competency Pillar Analysis

In [7]:
comp_merged = comp_yearly.merge(latest_perf[['employee_id','latest_rating']], on='employee_id')
comp_merged = comp_merged.merge(dim_pillars, on='pillar_code')

comp_pivot = (
    comp_merged.groupby(['pillar_label','latest_rating'])['score']
               .mean().reset_index()
               .pivot(index='pillar_label', columns='latest_rating', values='score')
)
print(comp_pivot.round(2))

fig = px.imshow(
    comp_pivot.T, title='Heatmap: Avg Competency Score by Pillar × Rating',
    labels={'x': 'Competency Pillar', 'y': 'Rating', 'color': 'Avg Score'},
    color_continuous_scale='Blues'
)
fig.show()
fig.write_image('../assets/charts/02_competency_heatmap.png')

latest_rating              0.0   1.0   2.0   3.0   4.0   5.0   6.0   99.0
pillar_label                                                             
Growth Drive & Resilience   2.0  1.15  1.94  3.76  4.01   4.9   1.0   4.0


## 6. Psychometric Profile: High vs Others

Using actual ERD columns: `iq`, `gtq`, `tiki`, `pauli`, `faxtor`

In [8]:
psych_merged = psych.merge(latest_perf[['employee_id','latest_rating']], on='employee_id')
psych_merged['group'] = psych_merged['latest_rating'].apply(lambda x: 'Rating 5' if x==5 else 'Others')

# Actual columns from ERD
numeric_cols = [c for c in ['iq','gtq','tiki','pauli','faxtor'] if c in psych_merged.columns]
print('Numeric columns available:', numeric_cols)

psych_group = psych_merged.groupby('group')[numeric_cols].mean().T
print('\nPsychometric averages by group:')
print(psych_group.round(2))

Numeric columns available: ['iq', 'gtq', 'tiki', 'pauli', 'faxtor']

Psychometric averages by group:
group   Others  Rating 5
iq      109.85    108.88
gtq      27.64     26.39
tiki      5.48      5.62
pauli    59.38     60.07
faxtor   59.93     61.84


In [9]:
# Radar chart — normalize each variable 0-100 for comparability
radar_cols = [c for c in ['iq','gtq','tiki','pauli','faxtor'] if c in psych_merged.columns]

# Min-max normalize per column
norm = psych_group.copy()
for col in norm.columns:
    mn, mx = norm[col].min(), norm[col].max()
    if mx > mn:
        norm[col] = (norm[col] - mn) / (mx - mn) * 100

fig = go.Figure()
for grp in ['Rating 5', 'Others']:
    if grp not in norm.columns:
        continue
    vals = norm[grp].tolist()
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=radar_cols + [radar_cols[0]],
        fill='toself', name=grp
    ))

fig.update_layout(title='Radar: Psychometric Profile — Rating 5 vs Others (normalized)')
fig.show()
fig.write_image('../assets/charts/03_psych_radar.png')

## 7. PAPI Kostick Analysis

In [10]:
papi_merged = papi.merge(latest_perf[['employee_id','latest_rating']], on='employee_id')
papi_merged['group'] = papi_merged['latest_rating'].apply(lambda x: 'Rating 5' if x==5 else 'Others')

papi_pivot = (
    papi_merged.groupby(['scale_code','group'])['score']
               .mean().reset_index()
               .pivot(index='scale_code', columns='group', values='score')
)
papi_pivot['diff_R5_minus_Others'] = papi_pivot.get('Rating 5', 0) - papi_pivot.get('Others', 0)
print(papi_pivot.sort_values('diff_R5_minus_Others', ascending=False).round(3))

fig = px.bar(
    papi_pivot.reset_index().sort_values('diff_R5_minus_Others', ascending=False),
    x='scale_code', y='diff_R5_minus_Others',
    title='PAPI Scale Difference: Rating 5 vs Others (positive = R5 higher)',
    color='diff_R5_minus_Others', color_continuous_scale='RdBu'
)
fig.show()
fig.write_image('../assets/charts/04_papi_diff.png')

group       Others  Rating 5  diff_R5_minus_Others
scale_code                                        
Papi_N       4.977     5.173                 0.196


## 8. CliftonStrengths — Top Themes Among Rating 5

In [11]:
str_merged = strengths_df.merge(latest_perf[['employee_id','latest_rating']], on='employee_id')

# Signature strengths = rank 1-5
top5_r5 = str_merged[(str_merged['latest_rating']==5) & (str_merged['rank']<=5)]
top5_others = str_merged[(str_merged['latest_rating']!=5) & (str_merged['rank']<=5)]

theme_r5     = top5_r5['theme'].value_counts().head(15).reset_index(name='count_r5')
theme_others = top5_others['theme'].value_counts().head(15).reset_index(name='count_others')
theme_comp   = theme_r5.merge(theme_others, on='theme', how='outer').fillna(0)
theme_comp   = theme_comp.sort_values('count_r5', ascending=False)

fig = px.bar(
    theme_comp.head(12), x='count_r5', y='theme', orientation='h',
    title='Top CliftonStrengths Themes — Rating 5 Employees (rank 1-5)',
    color='count_r5', color_continuous_scale='Blues'
)
fig.show()
fig.write_image('../assets/charts/05_strengths_top.png')

top_themes_list = theme_comp['theme'].head(5).tolist()
print('Top themes for Success Formula:', top_themes_list)

Top themes for Success Formula: ['Futuristic', 'Achiever', 'Command', 'Individualization', 'Context']


## 9. Contextual Factors

In [12]:
emp_merged = employees.merge(latest_perf[['employee_id','latest_rating']], on='employee_id')
emp_merged = emp_merged.merge(
    dim_grades.rename(columns={'name':'grade'})[['grade_id','grade']], on='grade_id', how='left'
)
emp_merged = emp_merged.merge(
    dim_edu.rename(columns={'name':'education'})[['education_id','education']], on='education_id', how='left'
)

fig = px.box(
    emp_merged, x='latest_rating', y='years_of_service_months',
    title='Years of Service (months) vs Performance Rating',
    color='latest_rating'
)
fig.show()
fig.write_image('../assets/charts/06_tenure_vs_rating.png')

grade_rating = emp_merged.groupby(['grade','latest_rating']).size().reset_index(name='count')
fig2 = px.bar(
    grade_rating, x='grade', y='count', color='latest_rating',
    barmode='group', title='Grade Distribution by Performance Rating'
)
fig2.show()
fig2.write_image('../assets/charts/07_grade_vs_rating.png')

## 10. Correlation Matrix — All Numerics vs Rating

In [13]:
master = emp_merged.merge(psych, on='employee_id', how='left')

num_cols = [c for c in
    ['latest_rating','years_of_service_months','iq','gtq','tiki','pauli','faxtor']
    if c in master.columns]

corr = master[num_cols].corr()
print('Correlation with rating:')
print(corr['latest_rating'].sort_values(ascending=False).round(3))

fig = px.imshow(
    corr, title='Correlation Matrix — Numeric Features',
    color_continuous_scale='RdBu', zmin=-1, zmax=1
)
fig.show()
fig.write_image('../assets/charts/08_correlation_matrix.png')

Correlation with rating:
latest_rating              1.000
faxtor                     0.033
gtq                        0.023
pauli                     -0.011
years_of_service_months   -0.020
tiki                      -0.033
iq                        -0.040
Name: latest_rating, dtype: float64


## 11. MBTI & DISC Distribution Among Rating 5

In [14]:
psych_full = psych_merged.copy()

# MBTI — normalize casing
if 'mbti' in psych_full.columns:
    psych_full['mbti_clean'] = psych_full['mbti'].str.strip().str.upper()
    mbti_r5 = psych_full[psych_full['latest_rating']==5]['mbti_clean'].value_counts().head(10)
    print('MBTI distribution — Rating 5:')
    print(mbti_r5)

# DISC
if 'disc' in psych_full.columns:
    disc_r5 = psych_full[psych_full['latest_rating']==5]['disc'].str.upper().value_counts().head(10)
    print('\nDISC distribution — Rating 5:')
    print(disc_r5)

MBTI distribution — Rating 5:
mbti_clean
ESTJ    7
INFP    6
ESTP    6
ESFJ    5
INTJ    5
ISFJ    4
ISTP    4
ESFP    3
INTP    2
ISFP    2
Name: count, dtype: int64

DISC distribution — Rating 5:
disc
ID    6
DC    6
SC    6
IS    5
CI    5
CS    4
SD    4
DS    4
IC    4
SI    3
Name: count, dtype: int64


## 12. Final Success Formula

Based on correlation analysis, group differences, and domain knowledge.

| TGV | Weight | TVs | Rationale |
|-----|--------|-----|-----------|
| Cognitive Ability | 30% | IQ, GTQ, TIKI, Pauli, Faxtor | Strongest predictor of performance ceiling |
| Competency Execution | 25% | All 10 pillar scores (avg) | Direct measure of role performance |
| Work Preferences (PAPI) | 20% | N,G,A,L,P scales; inverse Z,K | Work style alignment with top performers |
| Behavioral Strengths | 15% | Top CliftonStrengths theme match | Natural talent patterns |
| Contextual Fit | 10% | Years of service, grade | Organizational readiness |

**Match Formula:**
```
TV_match_rate = min(100, candidate_score / benchmark_median × 100)   [numeric, higher=better]
TV_match_rate = min(100, (2×benchmark - candidate) / benchmark × 100) [numeric, lower=better]
TGV_match_rate = avg(TV_match_rates within TGV)
Final_match_rate = Σ(TGV_match_rate × TGV_weight)
```


In [15]:
success_formula = {
    'Cognitive Ability':     {'weight': 0.30, 'tvs': ['iq','gtq','tiki','pauli','faxtor']},
    'Competency Execution':  {'weight': 0.25, 'tvs': ['All 10 pillar scores']},
    'Work Preferences':      {'weight': 0.20, 'tvs': ['Papi_N','Papi_G','Papi_A','Papi_L','Papi_P','Papi_Z(inv)','Papi_K(inv)']},
    'Behavioral Strengths':  {'weight': 0.15, 'tvs': ['CliftonStrengths top themes']},
    'Contextual Fit':        {'weight': 0.10, 'tvs': ['years_of_service_months']},
}

print('SUCCESS FORMULA')
print('='*60)
total_w = 0
for tgv, cfg in success_formula.items():
    print(f"  {tgv:28s} | {cfg['weight']:.0%} | {cfg['tvs']}")
    total_w += cfg['weight']
print(f"{'Total':28s} | {total_w:.0%}")

SUCCESS FORMULA
  Cognitive Ability            | 30% | ['iq', 'gtq', 'tiki', 'pauli', 'faxtor']
  Competency Execution         | 25% | ['All 10 pillar scores']
  Work Preferences             | 20% | ['Papi_N', 'Papi_G', 'Papi_A', 'Papi_L', 'Papi_P', 'Papi_Z(inv)', 'Papi_K(inv)']
  Behavioral Strengths         | 15% | ['CliftonStrengths top themes']
  Contextual Fit               | 10% | ['years_of_service_months']
Total                        | 100%
